# Notebook 11 - Structure-distance fixture builder

Builds the material the structure-distance experiments (E07-E11) consume: segmented statements, a byte-identical
reorder pool, natural pairs, and - new for E11 - a **second article** and a **true-paraphrase control**. Two
source articles are segmented into statements; gold-tier bases get a byte-identical reorder pool (the order-isolation
upper bound) and an opus-mt **back-translation paraphrase** (a per-statement reword, order preserved); same-model
**a/b regeneration pairs** add a second content-invariance control. Outputs land in `data/processed/structure-fixture/`.

- **Two articles** - IBM AI-adoption exec-summaries (`ibm`) and the Wergeland *Impact of AI on Society* curriculum (`wergeland`, extracted from the PDF)
- **Reorder pool** - `k`-swap permutations across a displacement grid, the byte-identical scramble upper bound, per base
- **Paraphrase control** - opus-mt EN->DE->EN per-statement reword (`~bt` variants) + same-model a/b regenerations; the content-invariance gate E10 could only proxy with embedding jitter
- **Section perturbation** - relocate statements across block boundaries

## GPU selection

This is a data-prep notebook - the only model is the `sat-3l-sm` statement segmenter (wtpsplit, CPU);
no embeddings are computed here. The GPU is pinned by UUID only so any lazy torch import lands on the
intended card, matching the E07 experiment notebook.

In [1]:
import os

# RTX 5000 Ada (GPU 2) by UUID - segmentation is CPU-light, this only pins a lazy torch import
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-c15a4c9a-8c2c-7fb9-a46b-fe4dff5dacf4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import warnings
warnings.filterwarnings("ignore")
print("fixture builder - segmentation only (CPU); GPU pinned to RTX 5000 Ada by UUID for consistency")

fixture builder - segmentation only (CPU); GPU pinned to RTX 5000 Ada by UUID for consistency


## Imports

In [2]:
%load_ext autoreload
%autoreload 2

import re, json, itertools
from pathlib import Path

import numpy as np
from rich.console import Console
from rich.table import Table

from docdistance.encoders import Segmenter

console = Console()
print("imports ready")

imports ready


## Reproducibility

In [3]:
SEED = 0
np.random.seed(SEED)
print(f"SEED={SEED}")

SEED=0


## Configuration

Paths, the 11-document manifest (same as E06), the reorder-sweep grid, and the section-perturbation
grid in one place. The reorder sweep generates a pool that E07 bins into 6 equal-width displacement
bins; the grid is sized so every bin holds >= 10 permutations after binning.

In [4]:
ROOT = Path("/home/lab/workspace/learning/projects/docdistance")
SUMMARY_DIR = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/summaries"
SOURCE_FILE = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/source/source-article.md"
WERG_PDF = ROOT / "data/external/FINAL-Impact-of-AI-on-society.pdf"
WERG_SOURCE = ROOT / "data/interim/ai-society-wergeland/source/source-article.md"
OUT_DIR = ROOT / "data/processed/structure-fixture"
PARA_DIR = ROOT / "data/interim/structure-paraphrase"          # cached back-translations
for _d in (OUT_DIR, PARA_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- article 1: IBM exec-summary fixture - (filename, label, tier); same set as E06 ---
DOCS = [
    ("exec-summary-gold-opus-4-8.md",     "gold",   "gold"),
    ("exec-summary-gold-2-opus-4-8.md",   "gold-2", "gold"),
    ("exec-summary-1-opus-4-8.md",        "v1",     "gold"),
    ("exec-summary-2-opus-4-8.md",        "v2",     "gold"),
    ("exec-summary-opus-4-8.md",          "opus",   "gold"),
    ("exec-summary-sonnet-4-6.md",        "sonnet", "gold"),
    ("exec-summary-haiku-4-5.md",         "haiku",  "gold"),
    ("exec-summary-adv1-a-sonnet-4-6.md", "adv1-a", "adv1"),
    ("exec-summary-adv1-b-sonnet-4-6.md", "adv1-b", "adv1"),
    ("exec-summary-adv2-a-haiku-4-5.md",  "adv2-a", "adv2"),
    ("exec-summary-adv2-b-haiku-4-5.md",  "adv2-b", "adv2"),
]
ANCHOR = "gold"

# --- article 2: Wergeland "Impact of AI on Society" - section groupings into bases (heading -> base) ---
WERG_BASES = {
    "werg-a": ["AI in Society"],
    "werg-b": ["Teaching the topic", "Bias and data"],
    "werg-c": ["Future"],
}

# same-model a/b regeneration pairs - real order-preserving paraphrases (content-invariance control, IBM)
AB_PAIRS = [("gold", "gold-2"), ("v1", "v2"), ("adv1-a", "adv1-b"), ("adv2-a", "adv2-b")]

# opus-mt back-translation (EN->DE->EN) - per-statement 1:1 reword, the content-invariance control for E11
BT_EN_DE = "Helsinki-NLP/opus-mt-en-de"
BT_DE_EN = "Helsinki-NLP/opus-mt-de-en"
BT_BEAMS = 5

# reorder/section bases - IBM gold-tier summaries + Wergeland bases (cross-article scramble)
IBM_BASES = ["gold", "gold-2", "v1", "v2", "opus", "sonnet", "haiku"]
WERG_BASE_LABELS = list(WERG_BASES)
BASE_DOCS = IBM_BASES + WERG_BASE_LABELS
SWAP_KS = [0, 1, 2, 3, 4, 6, 8, 11, 15, 20, 28, 40]   # number of random transpositions
SEEDS_PER_K = 14
N_BINS = 6                                            # E07/E11 bin displacement into this many

# section axis - relocate k statements across block boundaries
SECTION_K = [1, 2, 3]
SECTION_SEEDS = 10

t = Table(title="Notebook 11 configuration", title_style="bold cyan", show_header=False, box=None, padding=(0, 2))
t.add_column(style="bold cyan"); t.add_column()
t.add_row("Articles", "ibm (exec-summaries) + wergeland (AI-in-society sections)")
t.add_row("Documents", f"{len(DOCS)} IBM summaries + 1 IBM source + {len(WERG_BASES)} Wergeland bases")
t.add_row("Reorder bases", f"{len(BASE_DOCS)} ({', '.join(BASE_DOCS)})")
t.add_row("Swap grid", f"{SWAP_KS} x {SEEDS_PER_K} seeds + reversal anchors")
t.add_row("Paraphrase", f"opus-mt back-translation (EN-DE-EN, {BT_BEAMS} beams) + {len(AB_PAIRS)} a/b regen pairs")
t.add_row("Displacement bins", f"{N_BINS} equal-width on normalized footrule (binned in E07/E11)")
t.add_row("Section perturbation", f"relocate k in {SECTION_K}, {SECTION_SEEDS} seeds")
t.add_row("Output", str(OUT_DIR))
console.print(t)

                                          Notebook 11 configuration                                           
  Articles                ibm (exec-summaries) + wergeland (AI-in-society sections)                           
  Documents               11 IBM summaries + 1 IBM source + 3 Wergeland bases                                 
  Reorder bases           10 (gold, gold-2, v1, v2, opus, sonnet, haiku, werg-a, werg-b, werg-c)              
  Swap grid               [0, 1, 2, 3, 4, 6, 8, 11, 15, 20, 28, 40] x 14 seeds + reversal anchors             
  Paraphrase              opus-mt back-translation (EN-DE-EN, 5 beams) + 4 a/b regen pairs                    
  Displacement bins       6 equal-width on normalized footrule (binned in E07/E11)                            
  Section perturbation    relocate k in [1, 2, 3], 10 seeds                                                   
  Output                  /home/lab/workspace/learning/projects/docdistance/data/processed/structure-fixture

## Second article - Wergeland PDF extraction (provenance)

The Wergeland *Impact of AI on Society* curriculum is a two-column, design-heavy InDesign PDF (18 pages, born-digital
with a real text layer). `extract_text()` interleaves the two columns, so the reproducible extraction crops each page
into left/right columns, de-hyphenates, and dumps a raw text file. The **canonical** `source-article.md` used below
was hand-reviewed from that raw extract to drop floating definition/activity boxes, emoji exercises, and competence
lists - the same hand-curation the IBM `source-article.md` received. This cell re-creates the raw extract for
provenance; it does not overwrite the curated source.

In [5]:
import re as _re
import pdfplumber

def _dehyph(lines):
    out = []
    for ln in lines:
        if out and _re.search(r"[A-Za-z]-$", out[-1]):
            out[-1] = out[-1][:-1] + ln.lstrip()
        else:
            out.append(ln)
    return out

def _col_text(page, x0, x1):
    t = page.within_bbox((x0, 0, x1, page.height)).extract_text() or ""
    lines = [l.rstrip() for l in t.splitlines() if l.strip()]
    return "\n".join(_dehyph(lines))

def extract_two_column(pdf_path):
    """Per-column reading-order extraction for a two-column PDF -> list of (page, left, right)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            w, h, mid = page.width, page.height, page.width * 0.5
            pages.append((_col_text(page, 0, mid - 2), _col_text(page, mid + 2, w)))
    return pages

if WERG_PDF.exists():
    _raw = extract_two_column(WERG_PDF)
    _raw_path = WERG_SOURCE.parent / "raw-extract.txt"
    _raw_path.write_text("\n\n".join(f"[page {i} L]\n{L}\n[page {i} R]\n{R}" for i, (L, R) in enumerate(_raw)))
    print(f"raw per-column extract: {len(_raw)} pages -> {_raw_path} ({_raw_path.stat().st_size:,} bytes)")
else:
    print(f"PDF absent ({WERG_PDF.name}); run data/external/download-fixtures.py. Using curated source only.")
print(f"curated source (canonical): {WERG_SOURCE} (exists={WERG_SOURCE.exists()})")

raw per-column extract: 18 pages -> /home/lab/workspace/learning/projects/docdistance/data/interim/ai-society-wergeland/source/raw-extract.txt (29,761 bytes)
curated source (canonical): /home/lab/workspace/learning/projects/docdistance/data/interim/ai-society-wergeland/source/source-article.md (exists=True)


## Data loading and segmentation

Each document is split into blocks (paragraphs for summaries and Wergeland sections, headings for the IBM source),
then `sat-3l-sm` segments every block into statements; each statement carries its flattened order index plus its
block id and name. IBM summaries load from the manifest (`article=ibm`); the Wergeland bases are groups of curated
sections (`article=wergeland`). Every doc is tagged with its `article` so cross-summary pairs stay within an article.

In [6]:
def split_blocks(text, mode):
    """Return [(block_name, block_text)] - paragraphs (blank-line) or heading-sections."""
    if mode == "para":
        parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        return [(f"p{idx}", p) for idx, p in enumerate(parts)]
    # heading mode - a new block starts at each markdown heading line
    blocks, cur_name, cur = [], "preamble", []
    for line in text.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            if any(c.strip() for c in cur):
                blocks.append((cur_name, "\n".join(cur).strip()))
            cur_name = line.lstrip("# ").strip()[:48]
            cur = [line]
        else:
            cur.append(line)
    if any(c.strip() for c in cur):
        blocks.append((cur_name, "\n".join(cur).strip()))
    return blocks

seg = Segmenter()

def build_doc(text, mode):
    statements, block_id, block_name = [], [], []
    for bidx, (bname, btext) in enumerate(split_blocks(text, mode)):
        for s in seg.split(btext):
            statements.append(s); block_id.append(bidx); block_name.append(bname)
    return statements, block_id, block_name

def werg_sections(path):
    """Curated Wergeland markdown -> {heading: section_text}; strips the html comment and the title block."""
    raw = re.sub(r"<!--.*?-->", "", path.read_text(), flags=re.S)
    out = {}
    for chunk in raw.split("\n## ")[1:]:
        out[chunk.splitlines()[0].strip()] = "\n".join(chunk.splitlines()[1:]).strip()
    return out

DOCSTORE = {}
for filename, label, tier in DOCS:                       # article 1 - IBM summaries
    st, bid, bnm = build_doc((SUMMARY_DIR / filename).read_text(), "para")
    DOCSTORE[label] = {"statements": st, "block_id": bid, "block_name": bnm, "tier": tier,
                       "kind": "summary", "article": "ibm", "n": len(st), "n_blocks": len(set(bid))}
st, bid, bnm = build_doc(SOURCE_FILE.read_text(), "heading")
DOCSTORE["source"] = {"statements": st, "block_id": bid, "block_name": bnm, "tier": "source",
                      "kind": "source", "article": "ibm", "n": len(st), "n_blocks": len(set(bid))}

WSEC = werg_sections(WERG_SOURCE)                         # article 2 - Wergeland bases
for label, sections in WERG_BASES.items():
    text = "\n\n".join(WSEC[s] for s in sections)
    st, bid, bnm = build_doc(text, "para")
    DOCSTORE[label] = {"statements": st, "block_id": bid, "block_name": bnm, "tier": "gold",
                       "kind": "summary", "article": "wergeland", "n": len(st), "n_blocks": len(set(bid))}

t = Table(title="Segmented documents", title_style="bold cyan")
for c in ("label", "article", "tier", "kind", "statements", "blocks"):
    t.add_column(c, style="cyan" if c == "label" else None)
for label, d in DOCSTORE.items():
    style = "bold green" if label == ANCHOR else None
    t.add_row(label, d["article"], d["tier"], d["kind"], str(d["n"]), str(d["n_blocks"]), style=style)
console.print(t)
print(f"bases: {BASE_DOCS} | min base statements={min(DOCSTORE[b]['n'] for b in BASE_DOCS)}")

                      Segmented documents                      
┏━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┓
┃ label  ┃ article   ┃ tier   ┃ kind    ┃ statements ┃ blocks ┃
┡━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━┩
│ gold   │ ibm       │ gold   │ summary │ 14         │ 6      │
│ gold-2 │ ibm       │ gold   │ summary │ 14         │ 6      │
│ v1     │ ibm       │ gold   │ summary │ 13         │ 4      │
│ v2     │ ibm       │ gold   │ summary │ 12         │ 4      │
│ opus   │ ibm       │ gold   │ summary │ 14         │ 6      │
│ sonnet │ ibm       │ gold   │ summary │ 13         │ 6      │
│ haiku  │ ibm       │ gold   │ summary │ 15         │ 5      │
│ adv1-a │ ibm       │ adv1   │ summary │ 14         │ 5      │
│ adv1-b │ ibm       │ adv1   │ summary │ 9          │ 4      │
│ adv2-a │ ibm       │ adv2   │ summary │ 14         │ 5      │
│ adv2-b │ ibm       │ adv2   │ summary │ 12         │ 6      │
│ source │ ibm       │ source │ source  │ 62         │ 13     │
│ werg-a │ wergeland │ gold   │ summary │ 11         │ 5      │
│ werg-b │ wergeland │ gold   │ summary │ 13         │ 5      │
│ werg-c │ wergeland │ gold   │ summary │ 14         │ 3      │
└────────┴───────────┴────────┴─────────┴────────────┴────────┘

bases: ['gold', 'gold-2', 'v1', 'v2', 'opus', 'sonnet', 'haiku', 'werg-a', 'werg-b', 'werg-c'] | min base statements=11


## True-paraphrase control - opus-mt back-translation

The content-invariance gate E10 could only proxy with embedding jitter (cosine ~0.875). Here every base document's
statements are reworded 1:1 by **back-translation** (EN -> DE -> EN, opus-mt, deterministic beam search), preserving
statement count and reading order while changing surface form - validated at mmBERT cosine ~0.98, far more faithful
than jitter. Each base gets a `<base>~bt` variant (`kind=paraphrase`, same `block_id`/`n`); the structure distance
between a base and its `~bt` must read ~0. Back-translations are cached to `data/interim/structure-paraphrase/` so
re-runs are offline and instant.

In [7]:
import torch
from transformers import MarianMTModel, MarianTokenizer

os.environ.setdefault("HF_TOKEN", os.environ.get("HF_AUTH_TOKEN", ""))
_BT_DEV = "cuda" if torch.cuda.is_available() else "cpu"
_MT = {}

def _load_mt(name):
    if name not in _MT:
        tok = MarianTokenizer.from_pretrained(name)
        m = MarianMTModel.from_pretrained(name).to(_BT_DEV).eval()
        _MT[name] = (tok, m)
    return _MT[name]

@torch.no_grad()
def _translate(name, sents, beams=BT_BEAMS):
    tok, m = _load_mt(name)
    out = []
    for i in range(0, len(sents), 32):
        b = tok(sents[i:i + 32], return_tensors="pt", padding=True, truncation=True, max_length=512).to(_BT_DEV)
        g = m.generate(**b, num_beams=beams, max_length=512)
        out += [tok.decode(x, skip_special_tokens=True) for x in g]
    return out

def back_translate(sents):
    return _translate(BT_DE_EN, _translate(BT_EN_DE, sents))

n_gen, n_cache = 0, 0
for base in BASE_DOCS:
    src_st = DOCSTORE[base]["statements"]
    cache = PARA_DIR / f"{base}.bt.json"
    bt = json.loads(cache.read_text()) if cache.exists() else None
    if bt is None or len(bt) != len(src_st):             # missing or stale -> (re)generate
        bt = back_translate(src_st)
        cache.write_text(json.dumps(bt, ensure_ascii=False, indent=1)); n_gen += 1
    else:
        n_cache += 1
    d = DOCSTORE[base]
    DOCSTORE[f"{base}~bt"] = {"statements": bt, "block_id": d["block_id"], "block_name": d["block_name"],
                             "tier": d["tier"], "kind": "paraphrase", "article": d["article"],
                             "paraphrase_of": base, "n": len(bt), "n_blocks": d["n_blocks"]}

print(f"back-translation paraphrases: {len(BASE_DOCS)} bases ({n_gen} generated, {n_cache} cached)")
_ex = DOCSTORE["gold"]["statements"][0], DOCSTORE["gold~bt"]["statements"][0]
print(f"example reword:\n  orig: {_ex[0]}\n  ~bt : {_ex[1]}")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/258 [00:00<00:00, 41527.76it/s, Materializing param=final_logits_bias]

Loading weights:   0%|          | 1/258 [00:00<00:00, 387.46it/s, Materializing param=final_logits_bias]  

Loading weights:   1%|          | 2/258 [00:00<00:00, 353.01it/s, Materializing param=model.decoder.embed_positions.weight]

Loading weights:   1%|          | 2/258 [00:00<00:00, 279.92it/s, Materializing param=model.decoder.embed_positions.weight]

...

Loading weights: 100%|█████████▉| 257/258 [00:00<00:00, 893.78it/s, Materializing param=model.encoder.layers.5.self_attn_layer_norm.weight]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 893.78it/s, Materializing param=model.shared.weight]                               

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 893.78it/s, Materializing param=model.shared.weight]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 877.28it/s, Materializing param=model.shared.weight]

back-translation paraphrases: 10 bases (10 generated, 0 cached)
example reword:
  orig: # Executive Summary - Enterprise AI Adoption (Gold, Opus 4.8)
  ~bt : # Summary - Enterprise AI Adoption (Gold, Opus 4.8)


## Byte-identical reorder pool

The clean order-isolation fixture: a base document permuted into itself, so the transport plan `T` is
trivially sharp and the structure signal is pure order. Displacement is the normalized Spearman footrule
of the permutation; `k` random transpositions sweep it from 0 (identity) upward, reversal anchors fill
the top of the range. The pool is over-generated; E07 bins it into 6 equal-width bins and samples >= 10
per bin. This is the *upper bound* the design flags - the diffuse-`T` realism comes from the natural
cross-summary pairs below.

In [8]:
def norm_footrule(perm):
    """Normalized Spearman footrule of a permutation - 0 identity, ~1 reversal."""
    n = len(perm)
    if n < 2:
        return 0.0
    return float(np.abs(np.asarray(perm) - np.arange(n)).sum()) / (n * n // 2)

def k_swap(n, k, rng):
    p = np.arange(n)
    for _ in range(k):
        i, j = rng.integers(0, n), rng.integers(0, n)
        p[i], p[j] = p[j], p[i]
    return p

REORDER_POOL = {}
for bi, label in enumerate(BASE_DOCS):
    n = DOCSTORE[label]["n"]
    pool = []
    for ki, k in enumerate(SWAP_KS):
        for s in range(SEEDS_PER_K):
            rng = np.random.default_rng(SEED * 1_000_000 + bi * 10_000 + ki * 100 + s)
            p = k_swap(n, k, rng)
            pool.append({"k": int(k), "seed": int(s), "perm": p.tolist(), "disp": norm_footrule(p)})
    # reversal anchors - reverse then 0-2 swaps, to populate the high-displacement bin
    rev = np.arange(n)[::-1].copy()
    for s in range(SEEDS_PER_K):
        rng = np.random.default_rng(SEED * 1_000_000 + bi * 10_000 + 99 * 100 + s)
        p = rev.copy()
        for _ in range(int(rng.integers(0, 3))):
            i, j = rng.integers(0, n), rng.integers(0, n)
            p[i], p[j] = p[j], p[i]
        pool.append({"k": -1, "seed": int(s), "perm": p.tolist(), "disp": norm_footrule(p)})
    REORDER_POOL[label] = pool

# bin-occupancy check - confirm every one of the 6 bins holds >= 10 perms after binning
edges = np.linspace(0.0, 1.0, N_BINS + 1)
t = Table(title=f"Reorder pool - occupancy across {N_BINS} displacement bins", title_style="bold cyan")
t.add_column("base", style="cyan"); t.add_column("n", justify="right"); t.add_column("pool", justify="right")
for b in range(N_BINS):
    t.add_column(f"[{edges[b]:.2f},{edges[b+1]:.2f}]", justify="right")
ok = True
for label in BASE_DOCS:
    disps = np.array([e["disp"] for e in REORDER_POOL[label]])
    counts = [int(((disps >= edges[b]) & (disps < edges[b + 1] if b < N_BINS - 1 else disps <= edges[b + 1] + 1e-9)).sum())
              for b in range(N_BINS)]
    ok = ok and all(c >= 10 for c in counts)
    t.add_row(label, str(DOCSTORE[label]["n"]), str(len(disps)), *[str(c) for c in counts])
console.print(t)
print("every bin >= 10 perms:", ok, "| if a high bin is short, reversal anchors + small n cap it (noted in E07)")

                           Reorder pool - occupancy across 6 displacement bins                            
┏━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ base   ┃  n ┃ pool ┃ [0.00,0.17] ┃ [0.17,0.33] ┃ [0.33,0.50] ┃ [0.50,0.67] ┃ [0.67,0.83] ┃ [0.83,1.00] ┃
┡━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ gold   │ 14 │  182 │          40 │          28 │          32 │          40 │          25 │          17 │
│ gold-2 │ 14 │  182 │          42 │          33 │          28 │          38 │          24 │          17 │
│ v1     │ 13 │  182 │          33 │          31 │          29 │          44 │          26 │          19 │
│ v2     │ 12 │  182 │          35 │          25 │          25 │          48 │          34 │          15 │
│ opus   │ 14 │  182 │          44 │          20 │          36 │          36 │          29 │          17 │
│ sonnet │ 13 │  182 │          32 │          28 │          39 │          39 │          25 │          19 │
│ haiku  │ 15 │  182 │          44 │          21 │          31 │          42 │          31 │          13 │
│ werg-a │ 11 │  182 │          31 │          21 │          41 │          46 │          18 │          25 │
│ werg-b │ 13 │  182 │          35 │          30 │          25 │          54 │          24 │          14 │
│ werg-c │ 14 │  182 │          39 │          32 │          29 │          36 │          32 │          14 │
└────────┴────┴──────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

every bin >= 10 perms: True | if a high bin is short, reversal anchors + small n cap it (noted in E07)


## Natural pairs, paraphrase controls, and section perturbation

Four pair regimes. `cross_summary` is the realistic diffuse-T set, now grouped **per article**; `tier_contrast`
contrasts the IBM gold anchor against same- and different-content summaries; `paraphrase` (new) is the
content-invariance control - each base vs its back-translation, plus the same-model a/b regenerations - which must
read structure ~0; `section_swap` relocates statements across block boundaries (byte-identical statements, only the
block label moves) over every base.

In [9]:
summary_by_article = {}
for l, d in DOCSTORE.items():
    if d["kind"] == "summary":
        summary_by_article.setdefault(d["article"], []).append(l)

# realistic diffuse-T pairs, within each article
cross_summary = {art: [[a, b] for a, b in itertools.combinations(labels, 2)]
                 for art, labels in summary_by_article.items()}

# tier contrasts vs the gold anchor - IBM only (same content gold-tier vs different content adv tiers)
tier_contrast = []
for b in summary_by_article["ibm"]:
    if b == ANCHOR:
        continue
    tier_contrast.append({"a": ANCHOR, "b": b, "tier_b": DOCSTORE[b]["tier"],
                          "content": "same" if DOCSTORE[b]["tier"] == "gold" else "diff"})

# paraphrase regime - the content-invariance controls (must read structure ~0)
paraphrase = []
for base in BASE_DOCS:                                    # back-translation, both articles
    v = f"{base}~bt"
    if v in DOCSTORE:
        paraphrase.append({"a": base, "b": v, "kind": "backtranslation", "article": DOCSTORE[base]["article"]})
for a, b in AB_PAIRS:                                     # same-model a/b regenerations, IBM
    if a in DOCSTORE and b in DOCSTORE:
        paraphrase.append({"a": a, "b": b, "kind": "ab_regen", "article": "ibm"})

# section-swap - relocate k statements to a different block; byte-identical statements, only block label moves
def section_swap(block_id, k, rng):
    n = len(block_id)
    blocks = sorted(set(block_id))
    if len(blocks) < 2:
        return None
    idx = rng.choice(n, size=min(k, n), replace=False)
    new_block = list(block_id)
    for i in idx:
        others = [b for b in blocks if b != block_id[i]]
        new_block[int(i)] = int(rng.choice(others))
    return {"moved_idx": [int(x) for x in idx], "new_block_id": [int(x) for x in new_block]}

section_swaps = {}
for bi, label in enumerate(BASE_DOCS):
    d = DOCSTORE[label]
    if d["n_blocks"] < 2:
        continue
    entries = []
    for ki, k in enumerate(SECTION_K):
        for s in range(SECTION_SEEDS):
            rng = np.random.default_rng(SEED * 2_000_000 + bi * 10_000 + ki * 100 + s)
            sw = section_swap(d["block_id"], k, rng)
            if sw:
                sw.update({"k": int(k), "seed": int(s)})
                entries.append(sw)
    section_swaps[label] = entries

PAIRS = {
    "cross_summary": cross_summary,
    "tier_contrast": tier_contrast,
    "paraphrase": paraphrase,
    "section_summary_source": ["gold", "source"],
    "section_swap": section_swaps,
}
print(f"cross-summary: " + ", ".join(f"{a}={len(p)}" for a, p in cross_summary.items())
      + f" | tier contrasts: {len(tier_contrast)}")
print(f"paraphrase pairs: {len(paraphrase)} "
      f"(backtranslation={sum(p['kind']=='backtranslation' for p in paraphrase)}, "
      f"ab_regen={sum(p['kind']=='ab_regen' for p in paraphrase)})")
print(f"section-swap bases: {list(section_swaps)}")

cross-summary: ibm=55, wergeland=3 | tier contrasts: 10
paraphrase pairs: 14 (backtranslation=10, ab_regen=4)
section-swap bases: ['gold', 'gold-2', 'v1', 'v2', 'opus', 'sonnet', 'haiku', 'werg-a', 'werg-b', 'werg-c']


## Save fixture

In [10]:
meta = {
    "seed": SEED, "n_bins": N_BINS, "swap_ks": SWAP_KS, "seeds_per_k": SEEDS_PER_K,
    "base_docs": BASE_DOCS, "ibm_bases": IBM_BASES, "werg_bases": WERG_BASE_LABELS,
    "section_k": SECTION_K, "section_seeds": SECTION_SEEDS, "anchor": ANCHOR,
    "articles": {"ibm": "data/interim/exec-summaries/ibm-ai-adoption",
                 "wergeland": "data/interim/ai-society-wergeland"},
    "paraphrase": {"backtranslation": {"en_de": BT_EN_DE, "de_en": BT_DE_EN, "beams": BT_BEAMS},
                   "ab_regen_pairs": [list(p) for p in AB_PAIRS]},
    "block_unit": {"summary": "paragraph", "source": "heading", "wergeland": "paragraph"},
    "doc_counts": {l: {"n": d["n"], "n_blocks": d["n_blocks"], "tier": d["tier"],
                       "kind": d["kind"], "article": d.get("article")} for l, d in DOCSTORE.items()},
    "note": "two articles (IBM exec-summaries + Wergeland AI-in-society sections); reorder pool is the "
            "byte-identical scramble upper bound; paraphrase regime = opus-mt back-translation (per-statement "
            "1:1 reword, ~bt variants) + same-model a/b regenerations - the content-invariance controls "
            "replacing E10's embedding-jitter proxy",
}
(OUT_DIR / "statements.json").write_text(json.dumps(DOCSTORE, ensure_ascii=False, indent=1))
(OUT_DIR / "reorder_pool.json").write_text(json.dumps(REORDER_POOL, indent=1))
(OUT_DIR / "pairs.json").write_text(json.dumps(PAIRS, ensure_ascii=False, indent=1))
(OUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))

n_para = sum(1 for d in DOCSTORE.values() if d["kind"] == "paraphrase")
t = Table(title="Fixture written", title_style="bold green", show_header=True)
t.add_column("file", style="cyan"); t.add_column("bytes", justify="right"); t.add_column("contains")
t.add_row("statements.json", str((OUT_DIR / "statements.json").stat().st_size),
          f"{len(DOCSTORE)} docs ({n_para} ~bt paraphrases), 2 articles")
t.add_row("reorder_pool.json", str((OUT_DIR / "reorder_pool.json").stat().st_size),
          f"{sum(len(v) for v in REORDER_POOL.values())} permutations over {len(BASE_DOCS)} bases")
t.add_row("pairs.json", str((OUT_DIR / "pairs.json").stat().st_size),
          f"cross-summary + tier + paraphrase ({len(paraphrase)}) + section")
t.add_row("meta.json", str((OUT_DIR / "meta.json").stat().st_size), "seed, bins, grids, articles, paraphrase")
console.print(t)

                                 Fixture written                                 
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ file              ┃  bytes ┃ contains                                         ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ statements.json   │  69649 │ 25 docs (10 ~bt paraphrases), 2 articles         │
│ reorder_pool.json │ 321343 │ 1820 permutations over 10 bases                  │
│ pairs.json        │  68445 │ cross-summary + tier + paraphrase (14) + section │
│ meta.json         │   4836 │ seed, bins, grids, articles, paraphrase          │
└───────────────────┴────────┴──────────────────────────────────────────────────┘

## Summary

The fixture now carries two articles and a real paraphrase control. IBM exec-summaries (`article=ibm`) and the
Wergeland *Impact of AI on Society* sections (`article=wergeland`) are segmented into statements; every gold-tier
base has a byte-identical reorder pool and an opus-mt back-translation paraphrase (`~bt`), and the same-model a/b
regenerations add a second content-invariance control. E11 consumes `statements.json`, `reorder_pool.json`,
`pairs.json` (with the new `paraphrase` regime), and `meta.json` to decide the structure-distance SOTA (H44 vs H55)
on this material - cross-article and against a true reword, the gates E10 could not test.

In [11]:
import json as _json
need = ["statements.json", "reorder_pool.json", "pairs.json", "meta.json"]
assert all((OUT_DIR / f).exists() for f in need), "missing fixture file"
_st = _json.loads((OUT_DIR / "statements.json").read_text())
_pa = _json.loads((OUT_DIR / "pairs.json").read_text())
# every base has a matching back-translation of equal length, and the paraphrase regime is populated
for b in BASE_DOCS:
    assert f"{b}~bt" in _st, f"missing paraphrase for {b}"
    assert _st[f"{b}~bt"]["n"] == _st[b]["n"], f"paraphrase length mismatch for {b}"
assert len(_pa["paraphrase"]) >= len(BASE_DOCS), "paraphrase regime underpopulated"
assert set(_pa["cross_summary"]) == {"ibm", "wergeland"}, "cross_summary must be per-article"
print(f"OK - {len(_st)} docs, {len(BASE_DOCS)} bases each with a ~bt paraphrase, "
      f"{len(_pa['paraphrase'])} paraphrase pairs, cross_summary articles={list(_pa['cross_summary'])}")

OK - 25 docs, 10 bases each with a ~bt paraphrase, 14 paraphrase pairs, cross_summary articles=['ibm', 'wergeland']
